# Import

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sqpulse import (
    GaussianPulse,
    CosinePulse,
    LorentzianPulse,
    SquarePulse,
    DRAGPulse,
    FlatTopPulse,
    compare_pulses,
    spectral_leakage,
    Transmon,
    PulseSequence,
    Simulator,
    RabiExperiment,
    T1Experiment,
    RamseyExperiment,
)


# 一、脉冲波形与时频域特征分析

In [ ]:
p_gauss = GaussianPulse(duration=40e-9, amp=0.8, chop=6.0, drag=0, name="Gaussian")
fig = p_gauss.plot(domain="both")

In [ ]:
p_cos = CosinePulse(duration=40e-9, amp=1.0, drag=0, detune=0.0, name="Cosine (Hann)")
fig = p_cos.plot(domain="both")

In [ ]:
p_lorentz = LorentzianPulse(duration=40e-9, amp=1.0, gamma=10e-9, name="Lorentzian")
fig = p_lorentz.plot(domain="both")


In [ ]:
p_square = SquarePulse(duration=40e-9, amp=1.0, name="Square")
fig = p_square.plot(domain="both")

In [ ]:
p_flattop = FlatTopPulse(duration=40.0e-9, amp=1.0, ramp_time=4e-9, ramp_type="gaussian")
fig = p_flattop.plot(domain="both")

In [ ]:
p_drag = DRAGPulse(duration=40e-9, amp=1.0, drag=1, sigma=8e-9, name="Gaussian+DRAG")
fig = p_drag.plot(domain="both")

In [ ]:
fig_comp = compare_pulses([p_gauss, p_cos, p_lorentz], domain="both", dt=0.1e-9, freq_range=(-300e6, 300e6))

In [ ]:
# 计算 100 MHz 外的高频谱泄露比例
cutoff = 100e6
print(f"方波脉冲谱泄露 (>100 MHz):     {spectral_leakage(p_square, cutoff):.6f}")
print(f"洛伦兹脉冲谱泄露 (>100 MHz):   {spectral_leakage(p_lorentz, cutoff):.6f}")
print(f"高斯脉冲谱泄露 (>100 MHz):     {spectral_leakage(p_gauss, cutoff):.6f}")
print(f"余弦脉冲谱泄露 (>100 MHz):     {spectral_leakage(p_cos, cutoff):.6f}")

# 二、Transmon 物理模型与演化

In [ ]:
transmon = Transmon(
    name="q0",
    f_q=5e9,        # 5.0 GHz
    alpha=-250e6,    # -250 MHz
    levels=3,       # 3 能级 (|0>, |1>, |2>)
    t1=25e-6,     # T1 = 25 us
    t2=4e-6,     # T2 = 18 us
    omega_d=2*np.pi*120e6,
)
print(transmon)

In [ ]:
# 编排序列：一个高斯脉冲驱动 -> 延时 20 ns -> 另一个脉冲
seq = PulseSequence(name="sample_sequence")
seq.add(transmon.drive, GaussianPulse(duration=30e-9, amp=0.5))
seq.delay(transmon.drive, 20e-9)
seq.add(transmon.drive, CosinePulse(duration=30e-9, amp=0.4))

fig_seq = seq.plot(dt=0.1e-9)

In [ ]:
# 运行动力学主方程仿真
sim_res = Simulator.run(transmon, seq,dt=0.1e-9)
fig_dyn, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
sim_res.plot_populations(ax=ax1)
sim_res.plot_bloch_vector(ax=ax2)
plt.tight_layout()

# 三、测量实验模拟

## Rabi

In [ ]:
rabi_res = RabiExperiment.amplitude_rabi(
    transmon,
    pulse_type=GaussianPulse,
    duration=20e-9,
    amps=np.linspace(0.0, 0.9, 41),
    dt=1e-10,
)
print(f"标定所得 pi 脉冲幅度:   {rabi_res.amp_pi:.5f}")
print(f"标定所得 pi/2 脉冲幅度: {rabi_res.amp_pi_half:.5f}")
fig_rabi, ax_r = plt.subplots(figsize=(7, 4.5))
rabi_res.plot(ax=ax_r)
fig_rabi.tight_layout()

## T1

In [ ]:
pi_pulse = GaussianPulse(duration=20e-9, amp=rabi_res.amp_pi)
t1_res = T1Experiment.run(
    transmon,
    pi_pulse=pi_pulse,
    delays=np.linspace(0, 40e-6, 21),
    dt=1e-9,
)
print(f"设定理论 T1: {transmon.t1:.1f} ns | 拟合实测 T1: {t1_res.t1_fit:.1f} ns")
fig_t1, ax_t = plt.subplots(figsize=(7, 4.5))
t1_res.plot(ax=ax_t)
fig_t1.tight_layout()

## T2 Ramsey

In [ ]:
pi_half_pulse = GaussianPulse(duration=20e-9, amp=rabi_res.amp_pi_half)
detuning = 1e6  # 1 MHz 人为失谐
ramsey_res = RamseyExperiment.run(
    transmon,
    pi_half_pulse=pi_half_pulse,
    detuning=detuning,
    delays=np.linspace(0, 5e-6, 101),
    dt=1e-9,
)
print(f"设定理论 T2: {transmon.t2:.1f} ns | 拟合实测 T2*: {ramsey_res.t2_star:.1f} ns")
print(f"预设失谐:   {detuning*1e3:.2f} MHz   | 拟合实测失谐: {ramsey_res.fitted_detuning*1e3:.2f} MHz")
fig_ramsey, ax_m = plt.subplots(figsize=(7, 4.5))
ramsey_res.plot(ax=ax_m)
fig_ramsey.tight_layout()